In [7]:
import pandas as pd
import numpy as np

Filtering the kernel matrices by removing not needed proteins, reducing the dimension from $6112 \times 6112$ to $2318 \times 2318$. 

In [8]:
# Reads the input kernel matrix
#kernel_matrix = pd.read_excel("linear_kernel.xlsx", index_col=0)            #linear kernel matrix
#kernel_matrix = pd.read_excel("rbf_kernel.xlsx", index_col=0)              #rbf kernel matrix
kernel_matrix = pd.read_excel("diffusion_kernel.xlsx", index_col=0)        #diffusion kernel matrix

# Read the list of genes from the Excel file
genes_to_keep = pd.read_excel("genes_2318.xlsx", usecols=["Genes"])["Genes"].dropna().tolist()
genes_to_keep = [gene for gene in genes_to_keep if gene in kernel_matrix.index]

# Filter DataFrame
kernel_matrix_filtered = kernel_matrix.loc[genes_to_keep, genes_to_keep]

# Save the filtered kernel matrix to an Excel file
#kernel_matrix_filtered.to_excel("linear_kernel_matrix2318.xlsx")
#kernel_matrix_filtered.to_excel("rbf_kernel_matrix2318.xlsx")
kernel_matrix_filtered.to_excel("diffusion_kernel_matrix2318.xlsx")

The filtered kernel matrices are then centered and normalized.

In [9]:
def center_kernel(K):
    if isinstance(K, pd.DataFrame):
        K_values = K.values  # Convert DataFrame to NumPy array
    else:
        K_values = K  # Already a NumPy array

    row_mean = np.mean(K_values, axis=1, keepdims=True)  # Row mean (column vector)
    col_mean = np.mean(K_values, axis=0, keepdims=True)  # Column mean (row vector)
    overall_mean = np.mean(K_values)  # Single value (overall mean)

    R = K_values - row_mean - col_mean + overall_mean  # Apply centering formula

    # Convert back to DataFrame with original index/columns if needed
    if isinstance(K, pd.DataFrame):
        return pd.DataFrame(R, index=K.index, columns=K.columns)
    else:
        return R

def normalize_kernel(K):
    if not isinstance(K, pd.DataFrame):
        raise ValueError("Input kernel matrix must be a pandas DataFrame.")

    # Extract diagonal elements and take sqrt
    diag_K = np.sqrt(np.diag(K.values))
    diag_inv = np.reciprocal(diag_K, where=diag_K != 0)  # Avoid division by zero
    D_inv_sqrt = np.diag(diag_inv)  # Construct diagonal matrix

    # Normalize the kernel matrix
    KN_values = D_inv_sqrt @ K.values @ D_inv_sqrt  

    # Return as DataFrame with the original row and column names
    return pd.DataFrame(KN_values, index=K.index, columns=K.columns)

# Execute

# Centering

#kernel_matrix = pd.read_excel("linear_kernel_matrix2318.xlsx", index_col=0)
#kernel_matrix = pd.read_excel("rbf_kernel_matrix2318.xlsx", index_col=0)
kernel_matrix = pd.read_excel("diffusion_kernel_matrix2318.xlsx", index_col=0)
#kernel_matrix_CN = center_kernel(kernel_matrix)

# Normalizing
kernel_matrix_CN = normalize_kernel(kernel_matrix)

# Save the normalized/centered kernel matrix to an Excel file

#kernel_matrix_CN.to_excel("linear_kernel_matrix2318_CN.xlsx")
#kernel_matrix_CN.to_excel("rbf_kernel_matrix2318_CN.xlsx")
kernel_matrix_CN.to_excel("diffusion_kernel_matrix2318_CN.xlsx")